# 유튜브 자막 수집

원본 `python/youtube/youtube_api.ipynb`, `youtube_styler_script.ipynb`, `youtube_vacuum_script.ipynb`의 검색·자막·메타데이터 수집 흐름을 공개용으로 정리한 코드입니다. API 키와 수집 원문은 저장소에 넣지 않습니다. 이 코드는 당시의 영상 ID 목록·수집 건수를 재현하지 않습니다.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi

# API 키 설정 (구글 API 사용을 위해 필요)
API_KEY = os.environ['YOUTUBE_API_KEY']
youtube = build('youtube', 'v3', developerKey=API_KEY)


In [ ]:
# 유튜브 영상 검색 함수 (키워드로 검색)
def search_videos_by_keyword(keyword, start_date='2022-01-01T00:00:00Z',
                             end_date='2025-01-01T00:00:00Z', max_videos=100):
    video_ids = []
    page_token = None
    try:
        while len(video_ids) < max_videos:
            request = youtube.search().list(
                q=keyword, part='id', type='video',
                maxResults=min(50, max_videos - len(video_ids)),
                publishedAfter=start_date, publishedBefore=end_date,
                pageToken=page_token
            )
            response = request.execute()
            for item in response.get('items', []):
                video_ids.append(item['id']['videoId'])
            page_token = response.get('nextPageToken')
            if not page_token:
                break
    except Exception as e:
        print(f'영상 검색 중 오류 발생: {e}')
    return list(dict.fromkeys(video_ids))


# 한국어 자막 추출 함수
def get_korean_transcript(video_id):
    try:
        transcript = YouTubeTranscriptApi().fetch(video_id, languages=['ko'])
        return transcript
    except Exception as e:
        print(f'에러 발생: {e} (영상 ID: {video_id})')
        return None


# 영상 정보 가져오기 함수
def get_video_snippet(video_id):
    try:
        request = youtube.videos().list(part='snippet', id=video_id)
        response = request.execute()
        return response['items'][0]['snippet']
    except Exception as e:
        print(f'영상 정보 가져오기 실패: {e} (영상 ID: {video_id})')
        return None


In [ ]:
# 자막 데이터를 CSV로 저장하는 함수
def save_transcript_to_csv(video_ids, filename):
    results = []
    for video_id in video_ids:
        print(f'추출 중: {video_id}')
        transcript = get_korean_transcript(video_id)
        snippet = get_video_snippet(video_id)
        if not transcript or not snippet:
            continue

        full_transcript = ' '.join([entry.text for entry in transcript])
        results.append({
            'title': snippet['title'],
            'content': full_transcript,
            'date': snippet['publishedAt'].split('T')[0],
            'link': f'https://www.youtube.com/watch?v={video_id}',
            'platform': 'youtube'
        })

    result = pd.DataFrame(results, columns=['title', 'content', 'date', 'link', 'platform'])
    result = result.drop_duplicates(subset='link')
    result.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f'{filename} 파일이 저장되었습니다. ({len(result)}건)')
    return result


In [ ]:
# 검색어와 저장 파일명은 분석 대상에 맞게 변경
keyword = '#LG무선청소기'
data_path = Path(os.environ['LG_VOC_PRIVATE_DATA'])
output_folder = data_path / 'youtube'
output_folder.mkdir(parents=True, exist_ok=True)
video_ids = search_videos_by_keyword(keyword)
result = save_transcript_to_csv(video_ids, output_folder / 'lg_vacuum_script.csv')
